In [1]:
import os
import glob
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import xarray as xr
from tqdm import tqdm



In [9]:
# Folder containing ET tiffs
et_folder = "../data/OpenET_Rasters/OpenET_Monthly_Full_Extent"

# Shapefile path
shp_path = "../data/Field_Boundaries/cafe_field_perisitance_stats.shp"

# Output CSV
output_csv = "../data/pixel_level_ET_timeseries.csv"


# --- 2. LOAD SHAPEFILE ---
print("Loading shapefile...")
fields = gpd.read_file(shp_path)

Loading shapefile...


In [10]:
# 1. Stack Rasters First (Crucial for temporal alignment)
print("Searching for rasters...")
all_tiffs = sorted(glob.glob(os.path.join(et_folder, "*.tif")))

valid_months = ["04", "05", "06", "07", "08", "09", "10"]

filtered_files = []
dates = []

# Improved regex: looks for ET_Monthly_YYYY_MM anywhere in the filename
pattern = re.compile(r'ET_Monthly_(\d{4})_(\d{2})')

for f in all_tiffs:
    filename = os.path.basename(f)
    match = pattern.search(filename)
    
    if match:
        year, month = match.groups()
        if month in valid_months:
            filtered_files.append(f)
            dates.append(pd.to_datetime(f"{year}-{month}-01"))

# --- SAFETY CHECK ---
if not filtered_files:
    print(f"Error: No files matched the pattern in {et_folder}")
    if all_tiffs:
        print(f"Sample filename found: {os.path.basename(all_tiffs[0])}")
    raise ValueError("No matching objects found to concatenate. Check your 'valid_months' or filename patterns.")

print(f"Stacking {len(filtered_files)} rasters into a DataCube...")
# Open as a single 3D object
da = xr.concat([rxr.open_rasterio(f, masked=True).squeeze() for f in filtered_files], 
               dim=pd.Index(dates, name="time"))

# 2. Extract Field-by-Field
records = []
fields_proj = fields.to_crs(da.rio.crs)

for idx, field in tqdm(fields_proj.iterrows(), total=len(fields_proj), desc="Extracting Fields"):
    try:
        # Clip the entire temporal cube to this field at once
        clipped = da.rio.clip([field.geometry], fields_proj.crs, drop=True, all_touched=True)
        
        if clipped.size == 0: 
            continue

        # Calculate Field Mean per timestamp for Relative ET
        field_means = clipped.mean(dim=["x", "y"])

        # Flatten spatial dimensions but keep time (Y, X, Time) -> (Pixel, Time)
        stacked = clipped.stack(pixel=("y", "x")).dropna("pixel", how="all")
        
        # Extract data for every valid pixel in this field
        for p_idx in range(stacked.pixel.size):
            pixel_series = stacked.isel(pixel=p_idx)
            x_coord = float(pixel_series.x)
            y_coord = float(pixel_series.y)
            
            # Extract each month's value for this pixel
            for t_idx, date in enumerate(dates):
                val = float(pixel_series.isel(time=t_idx))
                
                # Skip if specific month has no data for this pixel
                if np.isnan(val): 
                    continue
                
                # Calculate relative ET using that month's specific field average
                month_avg = float(field_means.isel(time=t_idx))
                rel_et = val / month_avg if month_avg != 0 else np.nan
                
                records.append({
                    "field_id": field.get('field_id', idx), 
                    "year": date.year,
                    "month": date.month,
                    "x": x_coord,
                    "y": y_coord,
                    "ET": val,
                    "relative_ET": rel_et
                })
    except Exception as e:
        # Silently skip fields with topological errors or no overlap
        continue

# 3. Finalize and Save
df = pd.DataFrame(records)
if not df.empty:
    df.to_csv(output_csv, index=False)
    print(f"Successfully saved {len(df)} pixel records to {output_csv}")
else:
    print("Processed completed, but no pixel-level data was extracted.")

Searching for rasters...
✅ Stacking 175 rasters into a DataCube...


Extracting Fields: 100%|██████████| 111/111 [02:50<00:00,  1.54s/it]


Successfully saved 1762775 pixel records to ../data/pixel_level_ET_timeseries.csv
